cd Desktop/Coding\ project/StockAnalysis ; git add . ; git commit -m "In progress" ; git push origin main

git repack -a -d -f --depth=250 --window=250

git reflog expire --all --expire=now ; git gc --prune=now --aggressive

In [ ]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_excel_filename, get_infix, get_volume5m_data, generate_end_dates, merge_stocks, stock_market
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import multiprocessing
import numpy as np
import pandas as pd
from pandas import ExcelWriter as EW
from plot import *
from scipy.stats import false_discovery_control, kendalltau, linregress, pearsonr, spearmanr, ttest_ind, wilcoxon
from stock_screener import check_conds_tech, check_conds_fund, EM_rating, get_stock_info, stoploss_target
from technicals import *
from tqdm import tqdm

# Connect to TradingView
from tvDatafeed import TvDatafeed, Interval
tv = TvDatafeed()

# Start of the program
start = dt.datetime.now()

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
# Get the MMFI data
mmfi_symbol = "MMFI"
mmfi_data = tv.get_hist(symbol=mmfi_symbol, exchange="INDEX", interval=Interval.in_daily, n_bars=10000)

# Capitalize column names
mmfi_data.columns = [col.capitalize() for col in mmfi_data.columns]

# Ensure the index is in pandas datetime format
mmfi_data.index = pd.to_datetime(mmfi_data.index)

# Rename the index from datetime to Datetime
mmfi_data.index.name = "Datetime"

# Keep only the date part but maintain datetime objects
mmfi_data.index = mmfi_data.index.normalize()

# Drop the Volume column
mmfi_data = mmfi_data.drop(columns=["Volume"])

# Save the data to a CSV file
mmfi_data.to_csv(f"Price data/{mmfi_symbol}_{current_date}.csv")

In [ ]:
# Get the S&P 500 data
sp500_df = get_df("^GSPC", current_date, max_period=True)

# Find bear markets
bear_df, bear_starts, bear_ends = find_bear_markets(sp500_df)
bear_bottoms = bear_df["Lowest Date"]
bear_bottoms = [pd.to_datetime(date) for date in bear_bottoms]

# Calculate the 252-day rolling z-score for MMFI
mmfi_data = calculate_zscore(mmfi_data, "Close", 252)

# Filter S&P 500 data to match MMFI data timeframe
sp500_df_filtered = sp500_df[sp500_df.index >= mmfi_data.index[0]]

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# Plot S&P 500 and MMFI on the top subplot
ax1.plot(sp500_df_filtered.index, sp500_df_filtered["Close"], label="S&P 500")
ax1.set_title("S&P 500 and MMFI")
ax1.set_ylabel("Price")

# Set the x limit
ax1.set_xlim([mmfi_data.index[0], mmfi_data.index[-1]])

# Plot the MMFI on the bottom subplot
ax2.plot(mmfi_data.index, mmfi_data["Close"], color="purple", label="MMFI")
ax2.set_xlabel("Date")
ax2.set_ylabel("Value")

# Filter bear_bottoms to match MMFI data timeframe
relevant_bear_bottoms = [bottom for bottom in bear_bottoms if bottom >= mmfi_data.index[0] and bottom <= mmfi_data.index[-1]]

# Add vertical lines for each bear market bottom to the S&P 500 plot
bear_bottom_zscores = []
for date in relevant_bear_bottoms:
    ax1.axvline(x=date, color="r", linestyle=":", alpha=0.7)
    
    # Find the closest date in mmfi_data to the bear end date
    closest_date = mmfi_data.index[mmfi_data.index.get_indexer([date], method="nearest")[0]]

    # Get the MMFI at that date
    mmfi = mmfi_data.loc[closest_date, "Close"]
    
    # Get the z-score at that date
    z_score = mmfi_data.loc[closest_date, "Close Z-Score"]
    bear_bottom_zscores.append((date, closest_date, mmfi, z_score))

# Add vertical lines when MMFI is below 10
for date in mmfi_data.index[mmfi_data["Close"] <= 10]:
    ax2.axvline(x=date, color="green", linestyle="--", alpha=0.5)

# Manually add a line for the legend
custom_lines = [Line2D([0], [0], color="r", linestyle=":", alpha=0.7), 
                 Line2D([0], [0], color="green", linestyle="--", alpha=0.5)]
# Custom labels for the legend
custom_labels = ["Bear Market Bottom", "MMFI < 10"]

# Get existing handles and labels
handles, labels = ax1.get_legend_handles_labels()
handles.extend(custom_lines)
labels.extend(custom_labels)

ax1.legend(handles, labels)
ax2.legend()

plt.tight_layout()
plt.show()

# Print the bear end dates and their corresponding z-scores
print("Bear Market Bottoms and MMFI Data:")
for date, closest_date, mmfi, z_score in bear_bottom_zscores:
    print(f"Bear Market End Date: {date}, Closest MMFI Date: {closest_date}, MMFI: {mmfi}, Z-Score: {z_score}")

In [ ]:
# Define return periods in trading days
return_periods = {
    "1W": pd.Timedelta(days=5),
    "1M": pd.Timedelta(days=21),
    "3M": pd.Timedelta(days=63),
    "6M": pd.Timedelta(days=126),
    "1Y": pd.Timedelta(days=252)
}

# Find all MMFI signals <= 10 at least one week apart
signals = mmfi_data.index[mmfi_data["Close"] <= 10]
filtered_signals = signals[(signals.to_series().diff().dt.days.isna()) | (signals.to_series().diff().dt.days > 7)]

# Compute SP500 returns after each signal
def get_returns(dt):
    row = {"Date": dt.date()}
    if dt in sp500_df.index:
        start = sp500_df.at[dt, "Close"]
        for label, delta in return_periods.items():
            future_date = dt + delta
            # Find the closest trading day on or after the target date
            future_idx = sp500_df.index.searchsorted(future_date)
            if future_idx < len(sp500_df):
                end = sp500_df.iloc[future_idx]["Close"]
                row[label] = end / start - 1
            else:
                row[label] = None
    else:
        for label in return_periods:
            row[label] = None
    return row

mmfi_sp500_returns_df = pd.DataFrame([get_returns(dt) for dt in filtered_signals]).set_index("Date")

# Calculate the average returns for all signals
avg_row = mmfi_sp500_returns_df.mean().to_frame().T
avg_row.index = ["Average"]

# Append the average row to the DataFrame
mmfi_sp500_returns_df = pd.concat([mmfi_sp500_returns_df, avg_row])
display(mmfi_sp500_returns_df)

In [ ]:
import akshare as ak
stock_hk_spot_em_df = ak.stock_hk_spot_em()

In [ ]:
# Get the price data of QQQ, TQQQ, and VIX
qqq_df = get_df("^IXIC", current_date)
tqqq_df = get_df("TQQQ", current_date)
vix_df = get_df("^VIX", current_date)

# Filter qqq_df to match the starting date of TQQQ
qqq_df_tqqq = qqq_df[qqq_df.index >= tqqq_df.index[0]]

# Calculate the CAGR of TQQQ
CAGR_tqqq = (tqqq_df["Close"].iloc[-1] / tqqq_df["Close"].iloc[0])**(1 / (len(tqqq_df) / 252)) - 1

# Calculate the CAGR of QQQ
CAGR_qqq = (qqq_df_tqqq["Close"].iloc[-1] / qqq_df_tqqq["Close"].iloc[0])**(1 / (len(qqq_df_tqqq) / 252)) - 1

# Calculate the CAGR ratio of TQQQ to QQQ
tqqq_factor = CAGR_tqqq / CAGR_qqq

# Calculate the percentage change of QQQ
qqq_df["Percent Change"] = qqq_df["Close"].pct_change()

# Calculate the closing prices of TQQQ
qqq_df["TQQQ Percent Change"] = qqq_df["Percent Change"] * tqqq_factor
qqq_df["TQQQ Close"] = (1 + qqq_df["TQQQ Percent Change"]).cumprod()

# Create the mock TQQQ dataframe
tqqq_mock_df = qqq_df[["TQQQ Close", "TQQQ Percent Change"]].copy()

# Copy the index from qqq_df
tqqq_mock_df.index = qqq_df.index

# Scale the mock TQQQ closing prices to match the most recent TQQQ close
tqqq_mock_df["TQQQ Close"] = tqqq_mock_df["TQQQ Close"] * (tqqq_df["Close"].iloc[-1] / tqqq_mock_df["TQQQ Close"].iloc[-1])
tqqq_mock_df.rename(columns={"TQQQ Close": "Close", "TQQQ Percent Change": "Percent Change"}, inplace=True)

In [ ]:
show = 252 * 3
stocks = ["GC=F", "SI=F", "HG=F"]
dfs = [get_df(stock, current_date, redownload=True) for stock in stocks]
metal_df = merge_stocks(stocks, dfs)
metal_df["Gold/Silver Ratio"] = metal_df["Close (GC=F)"] / metal_df["Close (SI=F)"]
metal_df["Gold/Copper Ratio"] = metal_df["Close (GC=F)"] / metal_df["Close (HG=F)"]

# Restrict the dataframe
metal_df = metal_df[-show:]

# Create a figure with two subplots: one for the metal prices, one for the ratios
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

# Plot the metal prices on the first subplot
close_goldfirst = metal_df["Close (GC=F)"].iloc[0]
close_silverfirst = metal_df["Close (SI=F)"].iloc[0]
close_copperfirst = metal_df["Close (HG=F)"].iloc[0]
ax1.plot(100 / close_goldfirst * metal_df["Close (GC=F)"], label="Gold (scaled)", color="gold")
ax1.plot(100 / close_silverfirst * metal_df["Close (SI=F)"], label="Silver (scaled)", color="silver")
ax1.plot(100 / close_copperfirst * metal_df["Close (HG=F)"], label="Copper (scaled)", color="peru")

# Set the label of the first subplot
ax1.set_ylabel("Price")

# Set the x limit of the first subplot
ax1.set_xlim(metal_df.index[0], metal_df.index[-1])

# Plot the ratios on the second subplot
goldsilver_ratio_first = metal_df["Gold/Silver Ratio"].iloc[0]
goldcopper_ratio_first = metal_df["Gold/Copper Ratio"].iloc[0]
ax2.plot(100 / goldsilver_ratio_first * metal_df["Gold/Silver Ratio"], color="silver", label="Gold/Silver Ratio")
ax2.plot(100 / goldcopper_ratio_first * metal_df["Gold/Copper Ratio"], color="peru", label="Gold/Copper Ratio")

# Set the y label of the second subplot
ax2.set_ylabel("Ratio wrt Gold")

# Set the x label
plt.xlabel("Date")

# Set the title
plt.suptitle(f"Metal prices comparison")

# Combine the legends and place them at the top subplot
handles, labels = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
handles += handles2
labels += labels2
ax1.legend(handles, labels)

# Adjust the spacing between subplots
plt.tight_layout()

# Save the plot
plt.savefig("Result/Figure/metalcf.png", dpi=300)    

# Show the plot
plt.show()